# Exploratory Data Analysis (EDA)

This notebook performs descriptive analysis and visualization of political violence patterns across Afghanistan and its neighbors.

**What to look for:**
- Baseline violence levels in each country (baseline burden)
- Temporal patterns and V-shaped trajectory in Afghanistan
- Synchronization or asynchronization between countries
- Geopolitical events that may explain fluctuations

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
import warnings
warnings.filterwarnings('ignore')

# Define countries
NEIGHBORS = ['Turkmenistan', 'Uzbekistan', 'Tajikistan', 'Iran', 'Pakistan', 'China']
PALETTE = {
    'Afghanistan': '#8B1E3F',
    'Turkmenistan': '#1D3557',
    'Uzbekistan': '#457B9D',
    'Tajikistan': '#2A9D8F',
    'Iran': '#E76F51',
    'Pakistan': '#F4A261',
    'China': '#6D597A'
}

print("✓ Libraries and variables loaded")

## Load Processed Data

Load the cleaned dataset from the data cleaning pipeline.

In [ ]:
# Load cleaned data
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)
data_dir = os.path.join(project_root, 'data')

try:
    data_path = os.path.join(data_dir, 'processed_spillover_data.csv')
    df_data = pd.read_csv(data_path)
    df_data['date'] = pd.to_datetime(df_data['date'])
    print(f"✓ Loaded data: {len(df_data)} rows × {len(df_data.columns)} columns")
    print(f"  Date range: {df_data['date'].min()} to {df_data['date'].max()}")
except Exception as e:
    print(f"Error loading data: {e}")
    print("Please run 01_data_cleaning.ipynb first")
    df_data = None

## Descriptive Statistics

**Interpretation guide:**
- **Mean**: Average violence level (higher = more violence)
- **Std Dev**: Variability over time
- **Min/Max**: Range of observed violence
- **Higher mean** suggests countries are in conflict zones or experience spillover

In [ ]:
# Descriptive statistics
if df_data is not None:
    # Get all country event columns
    event_cols = [col for col in df_data.columns if 'events' in col and 'tplus1' not in col and 'Afghanistan' not in col]
    
    stats = []
    for col in event_cols:
        country = col.replace('_events', '')
        stats.append({
            'Country': country,
            'Mean': df_data[col].mean(),
            'Std Dev': df_data[col].std(),
            'Min': df_data[col].min(),
            'Max': df_data[col].max(),
            'Total': df_data[col].sum()
        })
    
    # Add Afghanistan
    stats.append({
        'Country': 'Afghanistan',
        'Mean': df_data['Afghanistan_events'].mean(),
        'Std Dev': df_data['Afghanistan_events'].std(),
        'Min': df_data['Afghanistan_events'].min(),
        'Max': df_data['Afghanistan_events'].max(),
        'Total': df_data['Afghanistan_events'].sum()
    })
    
    stats_df = pd.DataFrame(stats)
    print("\n=== Descriptive Statistics ===")
    print(stats_df.to_string(index=False))
    
: '#a1212b',
                    'Neighbors Avg': '#2b3b4c'
    }
    
    # Calculate neighbor average
    neighbor_cols = [f'{n}_events_tplus1' for n in NEIGHBORS]
    neighbor_events_tplus1 = df_data[neighbor_cols].mean(axis=1)
    
    df_plot = pd.DataFrame({
        'date': df_data['date'],
        'Afghanistan_events': df_data['Afghanistan_events'],
        'Neighbor_Avg_tplus1': neighbor_events_tplus1
    })
    
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=df_plot['date'],
        y=df_plot['Afghanistan_events'],
        mode='lines+markers',
        name='Afghanistan events (t)',
        line=dict(color=color_palette['Afghanistan'], width=2.8),
        marker=dict(size=6)
    ))
    
    fig.add_trace(go.Scatter(
        x=df_plot['date'],
        y=df_plot['Neighbor_Avg_tplus1'],
        mode='lines+markers',
        name='Mean neighbor events (t+1)',
        line=dict(color=color_palette['Neighbors Avg'], width=2.4, dash='dot'),
        marker=dict(size=6)
    ))
    
    # Add annotation for lag
    fig.add_annotation(
        text='One-month lag alignment',
        x=df_plot['date'].iloc[3],
        y=560,
        showarrow=True,
        arrowhead=2,
        ax=60, ay=-35,
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor='#d7dee5'
    )
    
    fig.update_layout(
        title='Lagged Design: Afghanistan(t) vs Average Neighbor(t+1)',
        xaxis_title='Aligned Axis: Afghanistan (t) & Neighbors (t+1)',
        yaxis_title='Events',
        template='plotly_white',
        height=410,
        margin=dict(l=40, r=25, t=70, b=45),
        legend=dict(orientation='h', y=-0.2),
        hovermode='x unified'
    )
    
    fig.show()
    print("\n✓ Figure 1 displayed")

## Time Series: Afghanistan V-Shape Pattern

**Interpretation:**
- **High at start** (Jan-Feb 2020): Pre-negotiation fighting
- **Low dip** (Apr-May 2020): Post-Doha Agreement, COVID-19 lockdown, Ramadan ceasefire
- **Rebound** (Oct-Dec 2020): Renewed Taliban offensive, US withdrawal accelerates

In [ ]:
# Afghanistan time series
if df_data is not None:
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=df_data['date'],
        y=df_data['Afghanistan_events'],
        mode='lines+markers',
        name='Afghanistan events',
        line=dict(color='#8B1E3F', width=3),
        marker=dict(size=7)
    ))
    
    fig.update_layout(
        title='Afghanistan Political Violence Events by Month (2020)',
        xaxis_title='Month',
        yaxis_title='Events',
        template='plotly_white',
        height=430,
        margin=dict(l=40, r=25, t=70, b=45)
    )
    
    fig.show()
    print("\n✓ Figure 2 displayed: Afghanistan time series shows clear V-shape")

## Neighbor Baseline Violence Levels

**Key observation:**
- Pakistan (orange) shows significantly higher violence than other neighbors
- Most Central Asian neighbors (Turkmenistan, Uzbekistan, Tajikistan) have low baseline violence
- This baseline variation is important for interpreting spillover effects

In [ ]:
# Neighbor time series
if df_data is not None:
    fig = go.Figure()
    
    colors_neighbors = {
        'Turkmenistan': '#1D3557',
        'Uzbekistan': '#457B9D',
        'Tajikistan': '#2A9D8F',
        'Iran': '#E76F51',
        'Pakistan': '#F4A261',
        'China': '#6D597A'
    }
    
    for neighbor in NEIGHBORS:
        col_name = f'{neighbor}_events_tplus1'
        if col_name in df_data.columns:
            fig.add_trace(go.Scatter(
                x=df_data['date'],
                y=df_data[col_name],
                mode='lines+markers',
                name=neighbor,
                line=dict(color=colors_neighbors[neighbor], width=2.2),
                marker=dict(size=6)
            ))
    
    fig.update_layout(
        title='Neighboring Countries Baseline Events (Month t+1)',
        xaxis_title='Actual Month (t+1)',
        yaxis_title='Neighbor events (t+1)',
        template='plotly_white',
        height=470,
        margin=dict(l=40, r=25, t=70, b=80),
        legend=dict(orientation='h', y=-0.22),
        hovermode='x unified'
    )
    
    fig.show()
    print("\n✓ Figure 3 displayed: Neighbor baseline violence patterns")
    print("  Pakistan clearly dominates with 40-75 events per month")
    print("  Central Asian neighbors largely peaceful (0-10 events)")

## Summary Statistics

Overall data coverage and quality metrics.

In [ ]:
# Summary statistics
if df_data is not None:
    print("\n=== Data Completeness ===")
    print(f"Total observations: {len(df_data)}")
    print(f"Missing values: {df_data.isnull().sum().sum()}")
    print(f"Date range: {df_data['date'].min().strftime('%Y-%m-%d')} to {df_data['date'].max().strftime('%Y-%m-%d')}")
    print(f"Number of months: {len(df_data)}")
    
    print("\n=== Country Coverage ===")
    agg_cols = [col for col in df_data.columns if 'events' in col and 'tplus1' not in col]
    for col in agg_cols:
        total = df_data[col].sum()
        avg = df_data[col].mean()
        country_name = col.replace('_events', '')
        print(f"{country_name:15} - Total: {total:4.0f} | Avg: {avg:5.1f} | Peak: {df_data[col].max():4.0f}")

# Exploratory data analysis

This notebook checks structure, summary statistics, and exploratory plots for the lag-aligned wide table.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path('/Users/chenyilu/Desktop/3148上交文件夹')
OUT_DIR = BASE_DIR / 'outputs'
FIG_DIR = OUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

data_path = OUT_DIR / 'afghanistan_neighbors_analysis_wide_2020.csv'
df = pd.read_csv(data_path)
df['DATE_TS'] = pd.to_datetime(df['DATE'] + '-01')
neighbor_cols = [c for c in df.columns if c.endswith('_events_tplus1')]
df.head()

## Exploratory Data Analysis (EDA)

This notebook provides descriptive statistics and visual overviews of the cleaned dataset before running formal statistical tests.

**Purpose**: 
- Understand the data's basic characteristics (min, max, mean, distribution)
- Identify patterns, trends, and anomalies
- Check data quality and completeness
- Visualize temporal trends for each country

**What to look for**:
- Which countries have high vs. low baseline violence levels?
- Are there seasonal or temporal patterns?
- How much variation is there within each country?
- Do countries have similar or different trend patterns?

In [ ]:
print('Shape:', df.shape)
print('Columns:', list(df.columns))
print('Missing values by column:')
display(df.isna().sum())
display(df.describe(include='all'))

### Descriptive Statistics

Display summary statistics for Afghanistan and each neighbor country:
- **Count**: Number of months with data (should be 12 for all countries)
- **Mean**: Average monthly violence count
- **Std**: Standard deviation (how much month-to-month variation exists)
- **Min/Max**: Range of values observed
- **25%/50%/75%**: Quartiles showing distribution shape

**Interpretation**:
- Higher mean = country tends to have more violence events
- Higher std = more volatile/unpredictable violence patterns
- Countries with similar means may have very different patterns (e.g., steady vs. erratic)

In [ ]:
plt.figure(figsize=(10, 4.8))
plt.plot(df['DATE_TS'], df['Afghanistan_events'], marker='o', linewidth=2)
plt.title('Afghanistan Events (2020)')
plt.xlabel('Month')
plt.ylabel('Events')
plt.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
path1 = FIG_DIR / 'eda_afghanistan_trend.png'
plt.savefig(path1, dpi=300)
plt.show()
print('Saved:', path1)

### Time Series Line Plots

Visualize how violence in each country changes across months in 2020.

**What to observe**:
- **Afghanistan**: Shows a V-shaped pattern (high → dips → rises again)
- **Neighbors**: Do they follow Afghanistan's pattern, or are they independent?
- **Overall trends**: Are there shared escalations or de-escalations?
- **Volatility**: Which countries have stable vs. volatile patterns?

**Geopolitical Context**:
- February 2020: US-Taliban agreement signed
- April-May 2020: Ramadan ceasefire period (explains the dip)
- Mid-2020 onward: Renewed fighting after ceasefire collapse

The time series reveals whether neighbors' violence patterns are synchronized with or independent of Afghanistan's.

In [ ]:
plt.figure(figsize=(12, 5.6))
for col in neighbor_cols:
    plt.plot(df['DATE_TS'], df[col], marker='o', linewidth=1.7, label=col.replace('_events_tplus1', ''))
plt.title('Neighbor Events at t+1 (Aligned to Afghanistan month t)')
plt.xlabel('Afghanistan month (t)')
plt.ylabel('Neighbor events (t+1)')
plt.legend(frameon=False, ncol=3)
plt.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
path2 = FIG_DIR / 'eda_neighbors_trend.png'
plt.savefig(path2, dpi=300)
plt.show()
print('Saved:', path2)

### Comparative Bar Charts

Compare total violence levels across countries for the full 2020 year.

**Key Takeaways**:
- **Afghanistan**: Far outpaces all neighbors (100+ more events per month on average)
- **Pakistan**: Highest among neighbors, substantially above all others
- **China, Iran**: Moderate levels, middle tier
- **Tajikistan, Uzbekistan, Turkmenistan**: Lowest levels, but still significant

**Interpretation for Spillover Analysis**:
- Countries with high baseline violence (Pakistan) may be affected differently than low-violence countries
- Geographic proximity and trade relationships may explain Pakistan's higher levels
- The substantial baseline differences across countries are important context for the regression analysis